In [84]:
import torch, os, torchaudio
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from BEATs import BEATs, BEATsConfig
#from sklearn.metrics import roc_auc_score, roc_curve

device = "cuda" if torch.cuda.is_available() else "cpu"

In [85]:
# load audio models

class UnlabeledDataset(Dataset):
    def __init__(self, folder, target_sr=16000):
        self.file_paths = []
        self.target_sr = target_sr

        for f in os.listdir(folder):
            if f.endswith(".wav") or f.endswith(".WAV"):
                self.file_paths.append(os.path.join(folder, f))

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        waveform, sr = torchaudio.load(self.file_paths[idx])

        # mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # resample
        if sr != self.target_sr:
            waveform = torchaudio.transforms.Resample(sr, self.target_sr)(waveform)

        waveform = waveform.squeeze(0)

        MAX_SECONDS = 5
        MAX_LENGTH = self.target_sr * MAX_SECONDS

        # pad / trim
        if waveform.shape[0] > MAX_LENGTH:
            waveform = waveform[:MAX_LENGTH]
        else:
            waveform = F.pad(waveform, (0, MAX_LENGTH - waveform.shape[0]))

        return waveform

def pad_audio_unlabelled(batch):
    waveforms = batch

    lengths = [w.shape[0] for w in waveforms]
    max_len = max(lengths)

    padded = []
    masks = []

    for w in waveforms:
        pad_len = max_len - w.shape[0]
        padded.append(F.pad(w, (0, pad_len)))

        mask = torch.zeros(max_len, dtype=torch.bool)
        if pad_len > 0:
            mask[-pad_len:] = True
        masks.append(mask)

    return torch.stack(padded), torch.stack(masks)
        

In [86]:
unlabelled_data = UnlabeledDataset(r'C:\Users\ASHU\Videos\Masters in IT\masters_lab\zataset')

test_loader = DataLoader(dataset=unlabelled_data, batch_size=8, shuffle=False, collate_fn=pad_audio_unlabelled)

In [87]:
class BEATsClassifier(nn.Module):
    def __init__(self, beats_model, num_classes=3):
        super().__init__()
        self.beats = beats_model
        self.classifier = nn.Linear(beats_model.cfg.encoder_embed_dim, num_classes)

    def forward(self, x, padding_mask=None):
        features = self.beats.extract_features(x, padding_mask)[0]  # (B, T, D)
        pooled = features.mean(dim=1)                               # (B, D)
        logits = self.classifier(pooled)
        return logits, pooled  

In [88]:
checkpoint = torch.load("BEATs_iter3_plus_AS2M.pt", map_location="cpu")
config = BEATsConfig(checkpoint["cfg"])
beats = BEATs(config)
beats.load_state_dict(checkpoint["model"])

model = BEATsClassifier(beats, num_classes=3).to(device)

state_dict = torch.load("beats_finetuned.pth", map_location=device)
model.load_state_dict(state_dict)
model.train()   # or model.train() for TTA

c:\Users\ASHU\anaconda3\envs\masters_env\lib\site-packages\torch\nn\utils\weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


BEATsClassifier(
  (beats): BEATs(
    (post_extract_proj): Linear(in_features=512, out_features=768, bias=True)
    (patch_embedding): Conv2d(1, 512, kernel_size=(16, 16), stride=(16, 16), bias=False)
    (dropout_input): Dropout(p=0.1, inplace=False)
    (encoder): TransformerEncoder(
      (pos_conv): Sequential(
        (0): Conv1d(768, 768, kernel_size=(128,), stride=(1,), padding=(64,), groups=16)
        (1): SamePad()
        (2): GELU(approximate='none')
      )
      (layers): ModuleList(
        (0): TransformerSentenceEncoderLayer(
          (self_attn): MultiheadAttention(
            (dropout_module): Dropout(p=0.1, inplace=False)
            (relative_attention_bias): Embedding(320, 12)
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768

In [89]:
missing, unexpected = model.load_state_dict(state_dict, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

Missing keys: []
Unexpected keys: []


In [90]:
# show layer names
for name, param in model.named_parameters():
    print(name)

beats.post_extract_proj.weight
beats.post_extract_proj.bias
beats.patch_embedding.weight
beats.encoder.pos_conv.0.bias
beats.encoder.pos_conv.0.weight_g
beats.encoder.pos_conv.0.weight_v
beats.encoder.layers.0.self_attn.grep_a
beats.encoder.layers.0.self_attn.relative_attention_bias.weight
beats.encoder.layers.0.self_attn.k_proj.weight
beats.encoder.layers.0.self_attn.k_proj.bias
beats.encoder.layers.0.self_attn.v_proj.weight
beats.encoder.layers.0.self_attn.v_proj.bias
beats.encoder.layers.0.self_attn.q_proj.weight
beats.encoder.layers.0.self_attn.q_proj.bias
beats.encoder.layers.0.self_attn.out_proj.weight
beats.encoder.layers.0.self_attn.out_proj.bias
beats.encoder.layers.0.self_attn.grep_linear.weight
beats.encoder.layers.0.self_attn.grep_linear.bias
beats.encoder.layers.0.self_attn_layer_norm.weight
beats.encoder.layers.0.self_attn_layer_norm.bias
beats.encoder.layers.0.fc1.weight
beats.encoder.layers.0.fc1.bias
beats.encoder.layers.0.fc2.weight
beats.encoder.layers.0.fc2.bias
bea

In [93]:
# freeze layers except classifier and layer11 for adaptation
for name, params in model.named_parameters():
    if "classifier" in name or "encoder.layers.11" in name:
        params.requires_grad = True
    else:
        params.requires_grad = False


In [94]:
# get model losses
def entropy_loss(p):
    return -(p * torch.log(p + 1e-8)).sum(dim=1).mean()

def norm_loss(feats): # rosetta inspired
    return feats.norm(p=2, dim=1).mean()

def ood_score(p):
    return -(p * torch.log(p + 1e-8)).sum(dim=1)

def angular_loss(feats, proto, labels):
    f = F.normalize(feats, dim=1)
    p = F.normalize(proto[labels], dim=1)
    return (1 - (f * p).sum(dim=1)).mean()



In [95]:
# prototypes
class Prototypes:
    def __init__(self, C, D, momentum=0.01):
        self.proto = torch.zeros(C,D).to(device)
        self.m = momentum

    @torch.no_grad()
    def update(self, feats, labels):
        for c in labels.unique():
            mask = labels == c
            if mask.sum() == 0:
                continue
            mean = feats[mask].mean(0)
            self.proto[c] = (1 - self.m) * self.proto[c] + self.m * mean


In [96]:
def split(probs, t=0.7):
    ent = -(probs * torch.log(probs + 1e-8)).sum(dim=1)
    return ent < t, ent >= t

In [97]:
# initialize
num_classes = model.classifier.out_features
print(f"Classes: {num_classes}")
feat_dim = model.classifier.in_features

prototypes = Prototypes(num_classes, feat_dim)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)
# OOD threshold
ood_threshold = 0.7
gamma1 = 0.5
gamma2 = 0.1

Classes: 3


In [98]:
# metrics tracking

all_preds = []
all_scores = []


In [99]:
running_loss = 0
num_batches = 0

print("\n===== STARTING TTA =====\n")

for audio, padding_mask in test_loader:

    audio = audio.to(device)
    padding_mask = padding_mask.to(device)

    logits, feats = model(audio, padding_mask)
    probs = F.softmax(logits, dim=1)

    csid_mask, csood_mask = split(probs)

    loss = 0

    # ---------------- ID adaptation ----------------
    if csid_mask.sum() > 0:
        f_id = feats[csid_mask]
        p_id = probs[csid_mask]
        pseudo_labels = p_id.argmax(dim=1)

        ent_loss = entropy_loss(p_id)
        ang_loss = gamma1 * angular_loss(f_id, prototypes.proto, pseudo_labels)

        loss += ent_loss + ang_loss

        prototypes.update(f_id.detach(), pseudo_labels)

    # ---------------- OOD regularization ----------------
    if csood_mask.sum() > 0:
        f_ood = feats[csood_mask]
        ood_loss = gamma2 * norm_loss(f_ood)
        loss += ood_loss
    else:
        ood_loss = torch.tensor(0.0, device=device)

    # ---------------- Backprop ----------------
    optimizer.zero_grad()
    loss.backward()
    if csid_mask.sum() > 0:
        optimizer.step()

    # ---------------- Predictions ----------------
    preds = probs.argmax(dim=1)
    score = ood_score(probs)

    final_preds = []
    unknown_count = 0

    for i in range(len(preds)):
        if score[i] > ood_threshold:
            final_preds.append(-1)
            unknown_count += 1
        else:
            final_preds.append(preds[i].item())

    # ---------------- METRICS ----------------
    batch_loss = loss.item()
    running_loss += batch_loss
    num_batches += 1

    known_count = len(preds) - unknown_count

    # ---------------- PRINTS ----------------
    print(f"\nBatch {num_batches}")
    print(f"Loss: {batch_loss:.4f}")
    print(f"Entropy loss: {ent_loss.item() if csid_mask.sum() > 0 else 0:.4f}")
    print(f"OOD loss: {ood_loss.item()}")
    print(f"Known predictions: {known_count}/{len(preds)}")
    print(f"Unknown (OOD): {unknown_count}/{len(preds)}")
    print(f"Mean OOD score: {score.mean().item():.4f}")

# ============================
# FINAL SUMMARY
# ============================

avg_loss = running_loss / num_batches

print("\n===== TTA FINISHED =====")
print(f"Average adaptation loss: {avg_loss:.4f}")
print(f"Total batches: {num_batches}")


===== STARTING TTA =====


Batch 1
Loss: 0.8005
Entropy loss: 0.3005
OOD loss: 0.0
Known predictions: 8/8
Unknown (OOD): 0/8
Mean OOD score: 0.3005

Batch 2
Loss: 0.9821
Entropy loss: 0.1335
OOD loss: 0.6480798721313477
Known predictions: 5/8
Unknown (OOD): 3/8
Mean OOD score: 0.3688

Batch 3
Loss: 0.2650
Entropy loss: 0.0284
OOD loss: 0.0
Known predictions: 8/8
Unknown (OOD): 0/8
Mean OOD score: 0.0284

Batch 4
Loss: 0.8253
Entropy loss: 0.1447
OOD loss: 0.3679118752479553
Known predictions: 6/8
Unknown (OOD): 2/8
Mean OOD score: 0.3314

Batch 5
Loss: 0.4258
Entropy loss: 0.2213
OOD loss: 0.0
Known predictions: 8/8
Unknown (OOD): 0/8
Mean OOD score: 0.2213

Batch 6
Loss: 0.2373
Entropy loss: 0.0575
OOD loss: 0.0
Known predictions: 8/8
Unknown (OOD): 0/8
Mean OOD score: 0.0575

Batch 7
Loss: 0.2480
Entropy loss: 0.0887
OOD loss: 0.0
Known predictions: 8/8
Unknown (OOD): 0/8
Mean OOD score: 0.0887

Batch 8
Loss: 0.2034
Entropy loss: 0.0373
OOD loss: 0.0
Known predictions: 8/8
Unknown (